In [ ]:
import numpy as np
import matplotlib.pyplot as plt

#### Returns generation function.

In [ ]:
def generate_log_returns(mean, variance):
    return np.random.normal(mean, variance)
    # return 0.001

#### State Variables
- Value in cash
- Value in stocks
- Total Budget
- Past price change direction
#### Action
- 10 => Shift 10% from cash to stock
- 0 => No Shift
- -10 => Shift 10% from stock to cash

In [ ]:
# Initial states
num_steps = 1000
start_price_change_dir = 0
value_cash = 500
value_stock = 500
total_pv = 1000

#### How action changes budget allocation.

In [ ]:
def alloc_change(curr_cash_val, curr_stock_val, action):
    pv = curr_cash_val + curr_stock_val
    cash_percent = (curr_cash_val/pv)*100
    stock_percent = (curr_stock_val/pv)*100
   
    new_cash = (cash_percent - action)*0.01*pv
    new_stock = (stock_percent + action)*0.01*pv
    if new_cash>=0 and new_cash<=pv:
        return(new_cash, new_stock)
    else:
        return(curr_cash_val, curr_stock_val)

### Creating environment class.

In [ ]:
class env_asset_allocation:
    def __init__(self, initial_cash, initial_stock, start_price_change_dir, num_steps=1000):
        self.init_cash = initial_cash
        self.init_stock = initial_stock
        self.init_past_mov_dir = start_price_change_dir
        self.init_steps = num_steps
        self.cash = initial_cash
        self.stock = initial_stock
        self.past_mov_dir = start_price_change_dir
        self.steps_remaining = num_steps
        self.state_observation = np.array([self.cash, self.stock, self.past_mov_dir])
        self.done = False
        
    def get_actions(self):
        return np.array([-0.05, 0, 0.05])
        
    def check_is_done(self):
        return (self.steps_remaining == 0)
        
    def action(self, action_value):
        if self.check_is_done():
            self.done = True
            # raise Exception("Reached the end of the simulation")
        # Cash and Stocks after changing allocation based on the action.
        cash_new, stock_new = alloc_change(self.cash, self.stock, action_value)
        
        # Stock value after price movement and some log return.
        log_return = generate_log_returns(-0.001, 0.001)
        stock_new = stock_new * np.exp(log_return)
        
        # Portfolio value (PV) initial and final
        PV_initial = self.cash + self.stock
        PV_final = cash_new + stock_new
        
        # Reward as the final returns.
        reward = PV_final - PV_initial
        
        # State update after taking the action.
        self.past_mov_dir = np.sign(log_return)
        self.cash = cash_new
        self.stock = stock_new
        self.steps_remaining-=1
        self.state_observation = np.array([self.cash, self.stock, self.past_mov_dir])
        return self.state_observation, reward, self.done, self.steps_remaining
        
    def reset(self):
        self.cash = self.init_cash
        self.stock = self.init_stock
        self.past_mov_dir = self.init_past_mov_dir
        self.steps_remaining = self.init_steps
        self.state_observation = [self.cash, self.stock, self.past_mov_dir]
        self.done = False
        return self.state_observation

### Creating agent class

In [ ]:
class Agent:
    def __init__(self):
        self.total_rewards=0.0
       
    def tabular_Q_agent(self, env:env_asset_allocation, num_episodes=1000, learning_rate=0.1, discount_factor=0.99, epsilon=0.1, plot=False):
        # Variable on which our action will be based
        past_mov_directions = np.array([-1,0,1])
        # Initialize Q-table with zeros
        Q = np.zeros((past_mov_directions.shape[0], env.get_actions().shape[0]))
        episode_returns_list = []
        Q0_list = []
        Q1_list = []
        Q2_list = []
        
        for episode in range(num_episodes):
            init_state = env.reset()
            done = False
            state_ind = np.where(past_mov_directions==init_state[-1])[0][0]
            cash_list = []
            stock_list =[]
            returns = 0
            while not done:
                # Choose action using epsilon-greedy policy
                if np.random.uniform(0, 1) < epsilon:
                    action_ind = np.random.choice([0,1,2])  # Explore
                else:
                    action_ind = np.argmax(Q[state_ind])  # Exploit
                    
                # Take action and observe reward and next state
                next_state, reward, done, _ = env.action([-0.05,0,0.05][action_ind])
                
                next_state_ind = np.where(past_mov_directions==next_state[-1])[0][0]
                # if next_state_ind==0 or next_state_ind==1:
                #     print(next_state)
                
                # Q-learning update rule
                best_next_action_ind = np.argmax(Q[next_state_ind])
                Q[state_ind, action_ind] = Q[state_ind, action_ind] + learning_rate * (
                    reward + discount_factor * Q[next_state_ind, best_next_action_ind] - Q[state_ind, action_ind]
                )
                
                returns = returns + 0.9*reward
                
                # Move to next state
                state_ind = next_state_ind
                cash_list.append(next_state[0])
                stock_list.append(next_state[1])
                
            print(returns)
            episode_returns_list.append(returns)
            cash_arr = np.array(cash_list)
            stock_arr = np.array(stock_list)
            pv_arr = cash_arr + stock_arr
            if plot == True:
                plt.plot(cash_arr/pv_arr, label='cash')
                plt.plot(stock_arr/pv_arr, label='stock')
                plt.legend()
                plt.savefig(f'img_QTable_neg/{episode}.png')
                plt.show()
                
            Q0_list.append(Q[2,0])
            Q1_list.append(Q[2,1])
            Q2_list.append(Q[2,2])
                
        return Q, Q0_list, Q1_list, Q2_list, episode_returns_list

In [ ]:
env = env_asset_allocation(500, 500, -1, 2000)
Q_agent = Agent()
Q, Q0_list, Q1_list, Q2_list, episode_returns_list = Q_agent.tabular_Q_agent(env=env, num_episodes=100, learning_rate=0.1, discount_factor=0.9, epsilon=0.1, plot=True)
Q

In [ ]:
print(episode_returns_list)

# Plotting cumulative episode reward
plt.plot(episode_returns_list)
plt.xlabel('Episode')
plt.ylabel('Reward')
plt.savefig('Returns_Qtable_neg')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
plt.plot(Q0_list, label='Q0')
plt.plot(Q1_list, label='Q1')
plt.plot(Q2_list, label='Q2')
plt.legend()
# plt.ylim([0,100000])
plt.show()

In [ ]:
episodes = 20
env = env_asset_allocation(500, 500, 0)
for episode in range(1, episodes+1):
    state = env.reset()
    done = False
    score = 0
    while not done:
        action = np.random.choice([-10, 0, 10])
        n_state, reward, done, remaining_steps = env.action(action)
        score+=reward
        # print(score)
    print(f'Episode:{episode}, Score:{score}')